# Module 03 — Lecture 2: Parallel LIF Simulation on GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_03_lif_neurons/02_parallel_lif_gpu.ipynb)

---

We now implement the LIF simulation on the GPU. The key insight:

> Each neuron is independent at every timestep → **one thread per neuron**.

**Learning objectives:**
- Implement a complete GPU LIF simulation with spike recording
- Use `atomicAdd` for safe concurrent spike logging
- Measure GPU throughput in neuron-steps/second
- Compare GPU vs CPU performance across network sizes

In [ ]:
!nvidia-smi

## 1. Design Decisions

Before writing code, three key design choices:

### Choice 1: SoA Memory Layout
Store state as **Structure of Arrays** (not Array of Structures):
```cpp
// GOOD (SoA): thread i reads V[i] — consecutive, coalesced
float V[N], m[N], h[N], n[N];

// BAD (AoS): thread i reads neurons[i].V — stride 4 floats, uncoalesced
struct Neuron { float V, m, h, n; } neurons[N];
```

### Choice 2: Constant Memory for Parameters
All neurons share τm, E_L, Rm, V_th, V_reset, dt → `__constant__` memory, free broadcast.

### Choice 3: Spike Recording with atomicAdd
Multiple threads may fire at the same timestep. To safely append to a shared spike log:
```cpp
int idx = atomicAdd(n_spikes, 1);  // atomically increment counter, get old value
spike_id[idx]   = neuron_id;       // safe: each thread gets unique idx
spike_time[idx] = current_time;
```

In [ ]:
%%writefile lif_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){ \
    fprintf(stderr,"CUDA: %s\n",cudaGetErrorString(e));exit(1);}} while(0)

// Parameters in constant memory
__constant__ float c_dt, c_tau_m, c_E_L, c_Rm, c_V_th, c_V_reset;
__constant__ int c_T_ref;

// ── Initialization kernel ─────────────────────────────────────────────────────
// Generates heterogeneous input currents using a simple hash RNG
__global__ void init_state(float* V, float* I, int* ref, int N,
                            float I_mean, float I_std, unsigned seed) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    V[i]   = c_E_L;   // start at rest
    ref[i] = 0;

    // Simple hash-based Gaussian (Box-Muller)
    unsigned u1 = (i * 1664525u + seed);
    unsigned u2 = (i * 22695477u + seed + 1);
    float r1 = (float)(u1 >> 9) / (float)(1 << 23);  // (0,1)
    float r2 = (float)(u2 >> 9) / (float)(1 << 23);
    r1 = fmaxf(r1, 1e-7f);  // avoid log(0)
    float z = sqrtf(-2.0f * logf(r1)) * cosf(6.2831853f * r2);
    I[i] = I_mean + I_std * z;
}

// ── LIF step kernel ───────────────────────────────────────────────────────────
__global__ void lif_step(float* V, const float* I, int* ref,
                          int* spike_id, float* spike_t,
                          int* n_spikes, int max_spikes,
                          int N, float t_ms) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    if (ref[i] > 0) {
        ref[i]--;
        V[i] = c_V_reset;
        return;
    }

    // Euler step
    float dV = c_dt / c_tau_m * (-(V[i] - c_E_L) + c_Rm * I[i]);
    V[i] += dV;

    if (V[i] >= c_V_th) {
        V[i] = c_V_reset;
        ref[i] = c_T_ref;

        // Atomically claim a slot in the spike log
        int idx = atomicAdd(n_spikes, 1);
        if (idx < max_spikes) {
            spike_id[idx] = i;
            spike_t[idx]  = t_ms;
        }
    }
}

int main(int argc, char** argv) {
    int   N      = (argc > 1) ? atoi(argv[1]) : 10000;
    float T_ms   = (argc > 2) ? atof(argv[2]) : 1000.0f;
    float dt     = 0.1f;
    int   T_steps= (int)(T_ms / dt);

    // Set constant memory
    float tau_m=20.f, E_L=-65.f, Rm=10.f, V_th=-55.f, V_reset=-70.f;
    int T_ref = (int)(2.0f / dt);
    CUDA_CHECK(cudaMemcpyToSymbol(c_dt,      &dt,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_m,   &tau_m,   sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_E_L,     &E_L,     sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_Rm,      &Rm,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_th,    &V_th,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_reset, &V_reset, sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_T_ref,   &T_ref,   sizeof(int)));

    size_t fb = N * sizeof(float), ib = N * sizeof(int);
    float *d_V, *d_I; int *d_ref;
    CUDA_CHECK(cudaMalloc(&d_V, fb)); CUDA_CHECK(cudaMalloc(&d_I, fb));
    CUDA_CHECK(cudaMalloc(&d_ref, ib));

    int max_spikes = N * 300;  // assume max 300 Hz × 1 s
    int *d_sid, *d_ns; float *d_st;
    CUDA_CHECK(cudaMalloc(&d_sid, max_spikes*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_st,  max_spikes*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_ns,  sizeof(int)));
    CUDA_CHECK(cudaMemset(d_ns, 0, sizeof(int)));

    int thr=256, blk=(N+thr-1)/thr;
    init_state<<<blk, thr>>>(d_V, d_I, d_ref, N, 1.8f, 0.3f, 42u);
    CUDA_CHECK(cudaDeviceSynchronize());

    cudaEvent_t t0, t1;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventRecord(t0));

    for (int step = 0; step < T_steps; step++) {
        lif_step<<<blk, thr>>>(d_V, d_I, d_ref,
                               d_sid, d_st, d_ns, max_spikes,
                               N, step * dt);
    }

    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    float ms; CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1));

    int h_ns;
    CUDA_CHECK(cudaMemcpy(&h_ns, d_ns, sizeof(int), cudaMemcpyDeviceToHost));
    h_ns = (h_ns < max_spikes) ? h_ns : max_spikes;

    int*   h_sid = (int*)malloc(h_ns*sizeof(int));
    float* h_st  = (float*)malloc(h_ns*sizeof(float));
    CUDA_CHECK(cudaMemcpy(h_sid, d_sid, h_ns*sizeof(int),   cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_st,  d_st,  h_ns*sizeof(float), cudaMemcpyDeviceToHost));

    printf("N=%d  T=%.0f ms  GPU=%.1f ms  throughput=%.1f M steps/s  spikes=%d  mean_fr=%.1f Hz\n",
           N, T_ms, ms,
           (float)N*T_steps/ms/1000.0f,
           h_ns, (float)h_ns/N/(T_ms/1000.0f));

    // Write spikes
    FILE* f = fopen("spikes.txt", "w");
    for (int k=0;k<h_ns;k++) fprintf(f, "%d %.2f\n", h_sid[k], h_st[k]);
    fclose(f);

    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(d_V);cudaFree(d_I);cudaFree(d_ref);
    cudaFree(d_sid);cudaFree(d_st);cudaFree(d_ns);
    free(h_sid); free(h_st);
    return 0;
}

In [ ]:
!nvcc -O2 -o lif_gpu lif_gpu.cu -lm

In [ ]:
# Benchmark: sweep N from 100 to 1M neurons
import subprocess, numpy as np, matplotlib.pyplot as plt

Ns = [100, 500, 1000, 5000, 10000, 50000, 100000, 500000]
results = []

for N in Ns:
    out = subprocess.run(['./lif_gpu', str(N), '1000'], capture_output=True, text=True).stdout
    print(out.strip())
    parts = out.split()
    try:
        gpu_ms = float(parts[4].split('=')[1])
        throughput = float(parts[5].split('=')[1])
        results.append((N, gpu_ms, throughput))
    except:
        pass

In [ ]:
if results:
    Ns_r, gpums, tput = zip(*results)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.loglog(Ns_r, [t * 1e6 for t in gpums], 'b-o', markersize=6)
    ax1.set_xlabel('Number of neurons (N)', fontsize=13)
    ax1.set_ylabel('GPU time per second of simulation (μs)', fontsize=12)
    ax1.set_title('LIF GPU Simulation Time', fontsize=13)
    ax1.grid(True, alpha=0.3)

    ax2.semilogx(Ns_r, tput, 'g-o', markersize=6)
    ax2.set_xlabel('Number of neurons (N)', fontsize=13)
    ax2.set_ylabel('Throughput (M neuron-steps/s)', fontsize=12)
    ax2.set_title('GPU Simulation Throughput', fontsize=13)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('lif_gpu_benchmark.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2. Understanding atomicAdd

When many threads fire spikes simultaneously, they all want to write to the shared spike log. Without atomics, two threads could read the same counter value and write to the same slot — data corruption.

```cpp
// WRONG — race condition if two threads fire simultaneously:
int idx = *n_spikes;     // thread A reads 42
                          // thread B also reads 42 (same time!)
*n_spikes = idx + 1;     // both write 43 — lost a spike!
spike_id[idx] = i;       // both write to [42] — one overwritten

// CORRECT — atomicAdd is an uninterruptible read-increment-write:
int idx = atomicAdd(n_spikes, 1);   // guaranteed unique idx per thread
spike_id[idx] = i;                  // each thread gets its own slot
```

**Performance note:** `atomicAdd` to global memory is slow (~100× slower than a register add). For large networks with high firing rates, this can become a bottleneck. Alternatives:
- Record only spike counts (one int per neuron, no atomics needed)
- Use per-block spike buffers with shared-memory atomics (fast), then merge
- Use a binary spike array `fired[N]` and compress offline

## Summary

| Design Choice | What We Did | Why |
|---------------|-------------|-----|
| Thread mapping | 1 thread per neuron | Neurons are independent per step |
| Memory layout | SoA (V[N], I[N]) | Coalesced access |
| Parameters | `__constant__` | Broadcast for free |
| Spike logging | `atomicAdd` to global counter | Safe concurrent append |
| Refractory period | per-neuron `ref[i]` counter | Decrement each step |

**Next lecture:** Visualise the simulation output with raster plots and firing rate analysis.